In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
import gc
from datetime import timedelta

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df
import asyncio
from telegram import Bot
import datetime

# Токен бота
TOKEN = '7919586728:AAHFa09TNqisddYLofOVY4SfmIFCeVHzVVg'

def telegram_sendMessage(chat_id, text):
    now = datetime.datetime.now()
    current_date = now.strftime('%d.%m.%Y')
    current_time = now.strftime('%H:%M:%S')

    async def send_message():
        bot = Bot(token=TOKEN)
        message = f"{text}\n🕒 Отправлено: {current_date} {current_time}"
        await bot.send_message(chat_id=chat_id, text=message)

    asyncio.run(send_message())

In [2]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [3]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [4]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="calamine",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул               Наименование Размер Коллекция       Бренд  \
0  00001851  Балетки женские 8L2139-1C     36       NaN  T.TACCARDI   
1  00001851  Балетки женские 8L2139-1C     37       NaN  T.TACCARDI   
2  00001851  Балетки женские 8L2139-1C     38       NaN  T.TACCARDI   
3  00001851  Балетки женские 8L2139-1C     39       NaN  T.TACCARDI   
4  00001851  Балетки женские 8L2139-1C     40       NaN  T.TACCARDI   

             Сезон    Направление Розничный отдел     Модель Бизнес-группа  \
0  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
1  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
2  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
3  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   
4  лето (закрытое)  Женская обувь   Женская обувь  8L2139-1C         Обувь   

   ... Техсегмент        

In [5]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               5
1  00001852               5
2  00001855               5
3  00001856               5
4  00001931               6
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 33.72 секунд


In [6]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2025-08-17  00006410               2
1  2025-08-17  00006630               1
2  2025-08-17  00006730               1
3  2025-08-17  00128845               4
4  2025-08-17  00146325               6
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 41.45 секунд


In [7]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2025-08-17  00006410     0.333333
1  2025-08-17  00006630     0.166667
2  2025-08-17  00006730     0.166667
3  2025-08-17  00128845     1.000000
4  2025-08-17  00146325     1.000000
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 35.39 секунд


In [8]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2025-08-17  00006410                   0
1  2025-08-17  00006630                   0
2  2025-08-17  00006730                   0
3  2025-08-17  00128845                 163
4  2025-08-17  00146325                 187
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 38.50 секунд


In [9]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2025-08-17  00006410                   0     0.333333
1  2025-08-17  00006630                   0     0.166667
2  2025-08-17  00006730                   0     0.166667
3  2025-08-17  00128845                 163     1.000000
4  2025-08-17  00146325                 187     1.000000
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 4.49 секунд


In [10]:
del df_stock

In [11]:
# 3. Получить данные из файлов вложенной папки "Показатели по дням"
try:
    print("Начинаем получать данные для Воронки...")
    start_time = time.time()  # Запускаем таймер
    folder_path_weeks = os.path.join(FOLDER_PATH, "Показатели по дням")
    df_funnel = pd.DataFrame()

    if os.path.exists(folder_path_weeks):
        
        for file in os.listdir(folder_path_weeks):
            file_path = os.path.join(folder_path_weeks, file)

            # Пропускаем скрытые файлы
            if is_hidden(file_path):
                print(f"Пропущен скрытый файл: {file}")
                continue

            # Проверяем расширение файла
            if file.endswith((".xlsx", ".xls")):
                try:
                    # Список столбцов, которые нужно взять из файла
                    columns_to_read = [
                        "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Продажи, ₽", "Заказы, шт", "Показы", "Клики",  "Цена"
                    ]

                    if file.endswith(".xlsx"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="calamine", usecols=columns_to_read)
                    elif file.endswith(".xls"):
                        temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="xlrd", usecols=columns_to_read)

                    # Переименование столбцов
                    temp_df.rename(columns={
                        "Артикул": "Артикул",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара",
                        "Заказы, шт": "Рекламные заказано товаров"
                    }, inplace=True, errors="ignore")

                    # Типы данных для столбцов
                    column_dtypes = {
                        "Артикул": str,
                        "Показы, всего": int,
                        "Показы на карточке товара": int,
                        "Показы в поиске и каталоге": int,
                        "Позиция в поиске и каталоге": float,
                        "В корзину, всего": int,
                        "Заказано товаров": int,
                        "Отменено товаров": int,
                        "Доставлено товаров": int,
                        "Возвращено товаров": int,
                        "Заказано на сумму": int,
                        "Выкупили ШТ": int,
                        "В корзину из карточки товара": int,
                        "ТипАктивности": str,
                        "Рекламные заказано на сумму": int,
                        "Рекламные заказано товаров": int,
                        "Рекламные показы на карточке товара": int,
                        "Рекламные показы": int,
                        "Расход, ₽": int,
                        "Цена": int
                    }

                    # Форматирование даты
                    temp_df = format_date_column(temp_df, 'Дата')

                    df_funnel = pd.concat([df_funnel, temp_df])
                except Exception as e:
                    print(f"Ошибка при чтении файла {file}: {e}")

        # Удаление лишних столбцов (если они остались)
        df_funnel = df_funnel[[
             "Дата",	"Артикул", "ТипАктивности", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
                        "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
                        "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
                        "Выкупили ШТ", "Расход, ₽", "Рекламные заказано на сумму", "Рекламные заказано товаров",
                        "Рекламные показы", "Рекламные показы на карточке товара", "Цена"
        ]]
        mask = pd.to_numeric(df_funnel['Расход, ₽'], errors='coerce').eq(0)
        df_funnel.loc[mask, 'ТипАктивности'] = 'Органика'

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Воронка:")
        print(df_funnel.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Воронки успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Папка 'Показатели по дням' не найдена.")
except Exception as e:
    print(f"Ошибка при получении данных для Воронки: {e}")

Начинаем получать данные для Воронки...
Первые 5 строк таблицы Воронка:
         Дата   Артикул ТипАктивности  Показы, всего  \
0  2025-09-01  023063J0      Органика              1   
1  2025-09-01  02306880      Органика              1   
2  2025-09-01  02306990      Органика              1   
3  2025-09-01  023063G0      Органика             53   
4  2025-09-01  b3106090      Органика              1   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                          0                           0   
1                          0                           0   
2                          0                           0   
3                          3                          33   
4                          0                           0   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                          NaN                 0                 0   
1                          NaN                 0                 0   
2                   

In [12]:
df_funnel

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Возвращено товаров,Заказано на сумму,В корзину из карточки товара,Выкупили ШТ,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Цена
0,2025-09-01,023063J0,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1556.0
1,2025-09-01,02306880,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,710.0
2,2025-09-01,02306990,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,813.0
3,2025-09-01,023063G0,Органика,53,3,33,51.85,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1113.0
4,2025-09-01,b3106090,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1620.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49495,2025-10-31,c0101130,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49496,2025-10-31,c1101020,Органика,3,1,1,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49497,2025-10-31,u4906230,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
49498,2025-10-31,NaN,Органика,1,0,0,NaN,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_funnel[df_funnel['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float64(17899191.29)

In [14]:
funnel_columns = df_funnel.columns

In [15]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, skiprows=1, engine='calamine')

    # оставляем только нужные колонки
    cols_keep = ["SKU", 'ID кампании', 'Инструмент', 'Место размещения',
       'Расход, ₽', 'Продажи, ₽',
       'Заказы, шт', 'Показы',
       'Клики']
    df_tmp = df_tmp[cols_keep]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed

    df_list.append(df_tmp)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ",
                        "Продажи, ₽": "Рекламные заказано на сумму",
                        "Показы": "Рекламные показы",
                        "Клики": "Рекламные показы на карточке товара", 
                        "Заказы, шт": "Рекламные заказано товаров"
                       },inplace=True)
df_all["Артикул OZ"] = df_all["Артикул OZ"].astype(str)

In [16]:
df_all#.drop_duplicates(subset=['Дата',"Артикул OZ"])

,Артикул OZ,ID кампании,Инструмент,Место размещения,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Дата
0,841850783,17708398,Оплата за клик,Поиск и рекомендации,117.948255,0.0,0,460,21,2025-09-30
1,1628955757,17330786,Оплата за клик,Поиск и рекомендации,0.000000,1428.0,1,0,0,2025-09-30
2,2366656419,17576553,Оплата за клик,Поиск и рекомендации,37.106727,0.0,0,33,4,2025-09-30
3,1151435994,17577580,Оплата за клик,Поиск и рекомендации,145.563666,0.0,0,1180,28,2025-09-30
4,1066494382,17558878,Оплата за клик,Поиск и рекомендации,139.571703,0.0,0,2452,35,2025-09-30
...,...,...,...,...,...,...,...,...,...,...
3106634,2310030361,18288265,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30
3106635,1649246458,18044269,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30
3106636,1267725462,17977104,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30
3106637,2659157998,18033552,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30


In [17]:
df_reference.columns

Index(['Артикул', 'Наименование', 'Размер', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Модель', 'Бизнес-группа', 'Группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Артикул OZ', 'Группа для отчетов', 'Себестоимость с НДС', 'НДС',
       'Процент выкупа', 'Ответственный за группу'],
      dtype='object')

In [18]:
df_all=df_all.merge(df_reference[['Артикул OZ','Артикул']], on='Артикул OZ', how='left')

In [19]:
# объединяем с df_funnel
# df_result = df_funnel_reference.merge(
#     df_all.drop_duplicates(subset=["Дата", "Артикул OZ"]),
#     on=["Дата", "Артикул OZ"],
#     how="left"
# )
df_all['ТипАктивности'] = pd.Series()

# Трафарет
mask = (
    ((df_all['Инструмент'] == 'Оплата за клик') &
    (df_all['Место размещения'] == 'Поиск и рекомендации'))
)
df_all.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Трафарет
mask = (
    (df_all['Инструмент'] == 'Оплата за заказ')
)
df_all.loc[mask, 'ТипАктивности'] = 'Оплата за заказ'

# Вывод в топ
mask = (
    ((df_all['Инструмент'] == 'Оплата за клик') & 
     (df_all['Место размещения'] == 'Поиск'))
)
df_all.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# Органика
mask = (
    ((df_all['Расход, ₽'] == 0) | 
     (df_all['Расход, ₽'] == 0.0))
)
df_all.loc[mask, 'ТипАктивности'] = 'Органика'

In [20]:
df_all['ТипАктивности'].unique()

array(['Трафарет', 'Органика', 'Вывод в топ', 'Оплата за заказ'],
      dtype=object)

In [21]:
df_all[df_all['ТипАктивности'].isna()]

,Артикул OZ,ID кампании,Инструмент,Место размещения,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Дата,Артикул,ТипАктивности


In [22]:
df_all.columns

Index(['Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Дата', 'Артикул',
       'ТипАктивности'],
      dtype='object')

In [23]:
funnel_columns = ['Дата', 'Артикул OZ', 'Артикул', 'ID кампании', 'Инструмент', 'Место размещения', 'ТипАктивности', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара'
       ]

In [24]:
# "Артикул": "Артикул",


In [25]:
# funnel_columns = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
#        'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
#        'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
#        'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
#        'В корзину из карточки товара', 'Выкупили ШТ', 
#        'Расход, ₽', 'Рекламные заказано на сумму',
#        'Рекламные заказано товаров', 'Рекламные показы',
#        'Рекламные показы на карточке товара', 'Цена'
#        ]

In [26]:
import pandas as pd
import numpy as np

def build_funnel_wide(
    df_raw: pd.DataFrame,
    funnel_columns: list,
    all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
    infer_organic_by_zero_spend=False,
    spend_col='Расход, ₽'
):
    """
    Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
    добавляет итоги и ПРОНОСИТ все прочие (не суммируемые) колонки из df_raw.
    """

    # 0) берём только нужные колонки для расчёта метрик (экономим память/время)
    cols_present = [c for c in funnel_columns if c in df_raw.columns]
    df = df_raw.loc[:, cols_present].copy()

    # 1) нормализуем типы активности
    type_col = 'ТипАктивности'
    # df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})

    if infer_organic_by_zero_spend and spend_col in df.columns:
        m = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
        df.loc[m, type_col] = 'Органика'

    # ключи и метрики
    key_cols = ['Дата', 'Артикул OZ']
    met_start = funnel_columns.index(type_col) + 1
    metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

    # привести метрики к числам (экономный тип)
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    # 2) суммируем по (Дата, Артикул, ТипАктивности)
    g = df.groupby(key_cols + [type_col], as_index=False)[metric_cols].sum(min_count=1)

    # база ключей (одна строка на Дата+Артикул)
    base = g[key_cols].drop_duplicates().set_index(key_cols).sort_index()

    # --- блоки метрик по каждому типу + базовые бинарные флаги ---
    metric_blocks, flag_blocks = [], []

    for t in all_types:
        sub = g[g[type_col] == t].set_index(key_cols)

        # метрики с префиксом
        if sub.empty:
            sub_metrics = pd.DataFrame(
                0.0, index=base.index,
                columns=[f'{t}_{m}' for m in metric_cols],
                dtype='float32'
            )
        else:
            sub_metrics = (sub[metric_cols]
                           .rename(columns={m: f'{t}_{m}' for m in metric_cols})
                           .reindex(base.index, fill_value=0.0)
                           .astype('float32'))
        metric_blocks.append(sub_metrics)

        # бинарный флаг наличия типа
        if sub.empty:
            flag = pd.Series(0, index=base.index, name=t, dtype='int8')
        else:
            flag = (pd.Series(1, index=sub.index, name=t)
                      .reindex(base.index, fill_value=0)
                      .astype('int8'))
        flag_blocks.append(flag)

    metrics_block = pd.concat(metric_blocks, axis=1)
    flags_block   = pd.concat(flag_blocks, axis=1)

    # --- 3) ДОП. КОЛОНКИ (все из df_raw, которых нет в funnel_columns) ---
    #    агрегируем по (Дата, Артикул) → first (первое ненулевое значение)
    extra_cols = [c for c in df_raw.columns
                  if c not in set(key_cols + [type_col] + metric_cols)]
    if extra_cols:
        # оставим только реально существующие
        extra_cols = [c for c in extra_cols if c in df_raw.columns]
        dims_block = (df_raw[key_cols + extra_cols]
                        .sort_values(key_cols)
                        .groupby(key_cols, as_index=False)
                        .first())  # берёт первое НЕ NaN
        # переиндексируем к базе
        dims_block = (dims_block.set_index(key_cols)
                                   .reindex(base.index)
                                   .reset_index())
    else:
        dims_block = base.reset_index()

    # --- 4) итоги без префиксов (сумма по всем типам) ---
    totals = pd.DataFrame(index=base.index)
    for m in metric_cols:
        totals[m] = metrics_block[[f'{t}_{m}' for t in all_types]].sum(axis=1).astype('float32')

    # --- 5) финальная сборка ---
    out = pd.concat(
        [
            dims_block,  # ключи + все доп. колонки
            flags_block.reset_index(drop=True),
            metrics_block.reset_index(drop=True),
            totals.reset_index(drop=True)
        ],
        axis=1
    ).copy()

    # --- 6) порядок колонок: ключи → доп.колонки → флаги → метрики по типам → итоги ---
    ordered = []
    # ключи
    ordered += key_cols
    # доп. колонки в исходном порядке df_raw (после ключей)
    ordered += [c for c in df_raw.columns
                if c in out.columns and c not in key_cols + [type_col] + metric_cols]
    # флаги типов
    ordered += [t for t in all_types if t in out.columns]
    # метрики по типам + итог без префикса
    for m in metric_cols:
        ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
        if m in out.columns:
            ordered += [m]

    out = out[[c for c in ordered if c in out.columns]]

    return out


In [27]:
df_all.columns

Index(['Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Дата', 'Артикул',
       'ТипАктивности'],
      dtype='object')

In [28]:
out = build_funnel_wide(df_raw=df_all, funnel_columns=funnel_columns)
out

,Дата,Артикул OZ,ID кампании,Инструмент,Место размещения,Артикул,Вывод в топ,Трафарет,Оплата за заказ,Органика,...,Вывод в топ_Рекламные показы,Трафарет_Рекламные показы,Оплата за заказ_Рекламные показы,Органика_Рекламные показы,Рекламные показы,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара
0,2025-09-03,1004068966,17007084,Оплата за клик,Поиск и рекомендации,W5255359,0,1,0,0,...,0.0,994.0,0.0,0.0,994.0,0.0,24.0,0.0,0.0,24.0
1,2025-09-03,1004068985,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,1,0,0,...,0.0,1354.0,0.0,0.0,1354.0,0.0,37.0,0.0,0.0,37.0
2,2025-09-03,1004069038,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,405.0,0.0,0.0,405.0,0.0,14.0,0.0,0.0,14.0
3,2025-09-03,1004069058,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,898.0,0.0,0.0,898.0,0.0,17.0,0.0,0.0,17.0
4,2025-09-03,1004069074,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,0,0,1,...,0.0,0.0,0.0,18.0,18.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919357,2025-11-16,998082254,18841446,Оплата за клик,Поиск и рекомендации,W6255274,0,1,0,0,...,0.0,2181.0,0.0,0.0,2181.0,0.0,91.0,0.0,0.0,91.0
2919358,2025-11-16,998082313,18841813,Оплата за клик,Поиск и рекомендации,D4155505,0,1,0,0,...,0.0,6.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,2.0
2919359,2025-11-16,998082323,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,252.0,0.0,0.0,252.0,0.0,8.0,0.0,0.0,8.0
2919360,2025-11-16,998082394,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,75.0,0.0,0.0,75.0,0.0,3.0,0.0,0.0,3.0


In [29]:
out.columns

Index(['Дата', 'Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения',
       'Артикул', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика',
       'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽',
       'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽',
       'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Рекламные заказано на сумму',
       'Оплата за заказ_Рекламные заказано на сумму',
       'Органика_Рекламные заказано на сумму', 'Рекламные заказано на сумму',
       'Вывод в топ_Рекламные заказано товаров',
       'Трафарет_Рекламные заказано товаров',
       'Оплата за заказ_Рекламные заказано товаров',
       'Органика_Рекламные заказано товаров', 'Рекламные заказано товаров',
       'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы',
       'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы',
       'Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара',
       'Трафарет_Рекламные показы на карточке това

In [30]:
df_all.drop_duplicates(subset=['Дата', 'Артикул OZ'])

,Артикул OZ,ID кампании,Инструмент,Место размещения,"Расход, ₽",Рекламные заказано на сумму,Рекламные заказано товаров,Рекламные показы,Рекламные показы на карточке товара,Дата,Артикул,ТипАктивности
0,841850783,17708398,Оплата за клик,Поиск и рекомендации,117.948255,0.0,0,460,21,2025-09-30,W5254734,Трафарет
1,1628955757,17330786,Оплата за клик,Поиск и рекомендации,0.000000,1428.0,1,0,0,2025-09-30,W4157136,Органика
2,2366656419,17576553,Оплата за клик,Поиск и рекомендации,37.106727,0.0,0,33,4,2025-09-30,M4159979,Трафарет
3,1151435994,17577580,Оплата за клик,Поиск и рекомендации,145.563666,0.0,0,1180,28,2025-09-30,W8355962,Трафарет
4,1066494382,17558878,Оплата за клик,Поиск и рекомендации,139.571703,0.0,0,2452,35,2025-09-30,W4155743,Трафарет
...,...,...,...,...,...,...,...,...,...,...,...,...
3106630,1624588243,18288814,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30,D2257440,Органика
3106632,1407483268,18032810,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30,D2156161,Органика
3106633,1095404817,17945339,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30,W7165536,Органика
3106634,2310030361,18288265,Оплата за клик,NaN,0.000000,0.0,0,0,0,2025-10-30,S7159103,Органика


In [31]:
print(list(out.columns))

['Дата', 'Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения', 'Артикул', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму', 'Трафарет_Рекламные заказано на сумму', 'Оплата за заказ_Рекламные заказано на сумму', 'Органика_Рекламные заказано на сумму', 'Рекламные заказано на сумму', 'Вывод в топ_Рекламные заказано товаров', 'Трафарет_Рекламные заказано товаров', 'Оплата за заказ_Рекламные заказано товаров', 'Органика_Рекламные заказано товаров', 'Рекламные заказано товаров', 'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы', 'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы', 'Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара', 'Трафарет_Рекламные показы на карточке товара', 'Оплата за заказ_Рекламные показы на карточке товара', 'Органика_Рекламные показы на карточке товара', 'Ре

In [32]:
df_all = out

In [33]:
df_all

,Дата,Артикул OZ,ID кампании,Инструмент,Место размещения,Артикул,Вывод в топ,Трафарет,Оплата за заказ,Органика,...,Вывод в топ_Рекламные показы,Трафарет_Рекламные показы,Оплата за заказ_Рекламные показы,Органика_Рекламные показы,Рекламные показы,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара
0,2025-09-03,1004068966,17007084,Оплата за клик,Поиск и рекомендации,W5255359,0,1,0,0,...,0.0,994.0,0.0,0.0,994.0,0.0,24.0,0.0,0.0,24.0
1,2025-09-03,1004068985,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,1,0,0,...,0.0,1354.0,0.0,0.0,1354.0,0.0,37.0,0.0,0.0,37.0
2,2025-09-03,1004069038,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,405.0,0.0,0.0,405.0,0.0,14.0,0.0,0.0,14.0
3,2025-09-03,1004069058,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,898.0,0.0,0.0,898.0,0.0,17.0,0.0,0.0,17.0
4,2025-09-03,1004069074,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,0,0,1,...,0.0,0.0,0.0,18.0,18.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919357,2025-11-16,998082254,18841446,Оплата за клик,Поиск и рекомендации,W6255274,0,1,0,0,...,0.0,2181.0,0.0,0.0,2181.0,0.0,91.0,0.0,0.0,91.0
2919358,2025-11-16,998082313,18841813,Оплата за клик,Поиск и рекомендации,D4155505,0,1,0,0,...,0.0,6.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,2.0
2919359,2025-11-16,998082323,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,252.0,0.0,0.0,252.0,0.0,8.0,0.0,0.0,8.0
2919360,2025-11-16,998082394,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,75.0,0.0,0.0,75.0,0.0,3.0,0.0,0.0,3.0


In [34]:
# example = out[out['Артикул'] == 'u5501030']
# example=example[example['Дата'] == "2025-08-29"]
# example.to_excel('Example Ozon.xlsx')

In [35]:
# 10. Связать "Воронка" с "Справочник"
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_funnel.")
            exit()

    # Список столбцов, которые нужно взять из справочника
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]

    # Фильтруем справочник, оставляя только нужные столбцы
    df_reference_filtered = df_reference[reference_columns]

    df_reference_filtered = df_reference_filtered.drop_duplicates(subset=['Артикул'])

    # Приводим типы данных к строковому формату
    df_funnel["Артикул"] = df_funnel["Артикул"].fillna('').astype(str).str[:8]  # Заменяем NaN на пустые строки
    df_reference_filtered["Артикул"] = df_reference_filtered["Артикул"].fillna('').astype(str).str[:8]

    # Объединение таблиц
    df_funnel_reference = pd.merge(
        df_funnel,
        df_reference_filtered,
        left_on="Артикул",
        right_on="Артикул",
        how="left"
    )

    # # Удаление дубликатов
    # df_funnel_reference = df_funnel_reference.drop_duplicates()

    # Удаление лишних столбцов (если они остались)
    # df_funnel_reference = df_funnel_reference.drop(columns=["Артикул WB"], errors="ignore")

    # Форматирование даты
    df_funnel_reference = format_date_column(df_funnel_reference, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

Начинаем создавать таблицу ВоронкаСправочник...
Первые 5 строк таблицы ВоронкаСправочник:
         Дата   Артикул ТипАктивности  Показы, всего  \
0  2025-09-01  023063J0      Органика              1   
1  2025-09-01  02306880      Органика              1   
2  2025-09-01  02306990      Органика              1   
3  2025-09-01  023063G0      Органика             53   
4  2025-09-01  b3106090      Органика              1   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                          0                           0   
1                          0                           0   
2                          0                           0   
3                          3                          33   
4                          0                           0   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                          NaN                 0                 0   
1                          NaN                 0                 0   
2 

In [36]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float64(17899191.29)

In [37]:
# del df_funnel

In [38]:
df_all#.drop_duplicates(subset=['Артикул OZ', 'Дата'])

,Дата,Артикул OZ,ID кампании,Инструмент,Место размещения,Артикул,Вывод в топ,Трафарет,Оплата за заказ,Органика,...,Вывод в топ_Рекламные показы,Трафарет_Рекламные показы,Оплата за заказ_Рекламные показы,Органика_Рекламные показы,Рекламные показы,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара
0,2025-09-03,1004068966,17007084,Оплата за клик,Поиск и рекомендации,W5255359,0,1,0,0,...,0.0,994.0,0.0,0.0,994.0,0.0,24.0,0.0,0.0,24.0
1,2025-09-03,1004068985,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,1,0,0,...,0.0,1354.0,0.0,0.0,1354.0,0.0,37.0,0.0,0.0,37.0
2,2025-09-03,1004069038,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,405.0,0.0,0.0,405.0,0.0,14.0,0.0,0.0,14.0
3,2025-09-03,1004069058,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,898.0,0.0,0.0,898.0,0.0,17.0,0.0,0.0,17.0
4,2025-09-03,1004069074,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,0,0,1,...,0.0,0.0,0.0,18.0,18.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919357,2025-11-16,998082254,18841446,Оплата за клик,Поиск и рекомендации,W6255274,0,1,0,0,...,0.0,2181.0,0.0,0.0,2181.0,0.0,91.0,0.0,0.0,91.0
2919358,2025-11-16,998082313,18841813,Оплата за клик,Поиск и рекомендации,D4155505,0,1,0,0,...,0.0,6.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,2.0
2919359,2025-11-16,998082323,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,252.0,0.0,0.0,252.0,0.0,8.0,0.0,0.0,8.0
2919360,2025-11-16,998082394,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,75.0,0.0,0.0,75.0,0.0,3.0,0.0,0.0,3.0


In [39]:
df_all[df_all['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float32(17899190.0)

In [40]:
# df_all[df_all['Дата'] == "2025-10-28"]['Показы'].sum()

In [41]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-10-28"]['Показы, всего'].sum()

np.int64(134062549)

In [42]:
df_all

,Дата,Артикул OZ,ID кампании,Инструмент,Место размещения,Артикул,Вывод в топ,Трафарет,Оплата за заказ,Органика,...,Вывод в топ_Рекламные показы,Трафарет_Рекламные показы,Оплата за заказ_Рекламные показы,Органика_Рекламные показы,Рекламные показы,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара
0,2025-09-03,1004068966,17007084,Оплата за клик,Поиск и рекомендации,W5255359,0,1,0,0,...,0.0,994.0,0.0,0.0,994.0,0.0,24.0,0.0,0.0,24.0
1,2025-09-03,1004068985,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,1,0,0,...,0.0,1354.0,0.0,0.0,1354.0,0.0,37.0,0.0,0.0,37.0
2,2025-09-03,1004069038,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,405.0,0.0,0.0,405.0,0.0,14.0,0.0,0.0,14.0
3,2025-09-03,1004069058,17257486,Оплата за клик,Поиск и рекомендации,W5255315,0,1,0,0,...,0.0,898.0,0.0,0.0,898.0,0.0,17.0,0.0,0.0,17.0
4,2025-09-03,1004069074,17008124,Оплата за клик,Поиск и рекомендации,W5255371,0,0,0,1,...,0.0,0.0,0.0,18.0,18.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2919357,2025-11-16,998082254,18841446,Оплата за клик,Поиск и рекомендации,W6255274,0,1,0,0,...,0.0,2181.0,0.0,0.0,2181.0,0.0,91.0,0.0,0.0,91.0
2919358,2025-11-16,998082313,18841813,Оплата за клик,Поиск и рекомендации,D4155505,0,1,0,0,...,0.0,6.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,2.0
2919359,2025-11-16,998082323,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,252.0,0.0,0.0,252.0,0.0,8.0,0.0,0.0,8.0
2919360,2025-11-16,998082394,18837931,Оплата за клик,Поиск и рекомендации,W5205365,0,1,0,0,...,0.0,75.0,0.0,0.0,75.0,0.0,3.0,0.0,0.0,3.0


In [43]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена', 'Артикул OZ',
       'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление',
       'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент',
       'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов'],
      dtype='object')

In [44]:
df_result = df_funnel_reference.drop(columns=['Расход, ₽','Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара',]).merge(
    df_all.drop(columns=["Артикул"]),
    on=["Дата", "Артикул OZ"],
    how="outer"
).drop_duplicates(subset=["Дата", "Артикул OZ"])

In [45]:
# объединяем с df_funnel
# df_result = df_funnel_reference.drop(columns=['Расход, ₽']).merge(
#     df_all.drop_duplicates(subset=["Дата", "Артикул OZ"]),
#     on=["Дата", "Артикул OZ"],
#     how="right"
# )

# # Трафарет
# mask = (
#     ((df_result['Инструмент'] == 'Оплата за клик') &
#     (df_result['Место размещения'] == 'Поиск и рекомендации')) |
#     (df_result['ТипАктивности'] == 'Оплата за клик')
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# # Вывод в топ
# mask = (
#     ((df_result['Инструмент'] == 'Оплата за клик') & 
#      (df_result['Место размещения'] == 'Поиск')) |
#     (df_result['ТипАктивности'] == 'ТОП')
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# # Органика
# mask = (
#     ((df_result['Расход, ₽'] == 0) | 
#      (df_result['Расход, ₽'] == 0.0))
# )
# df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [46]:
df_all.columns

Index(['Дата', 'Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения',
       'Артикул', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика',
       'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽',
       'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽',
       'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Рекламные заказано на сумму',
       'Оплата за заказ_Рекламные заказано на сумму',
       'Органика_Рекламные заказано на сумму', 'Рекламные заказано на сумму',
       'Вывод в топ_Рекламные заказано товаров',
       'Трафарет_Рекламные заказано товаров',
       'Оплата за заказ_Рекламные заказано товаров',
       'Органика_Рекламные заказано товаров', 'Рекламные заказано товаров',
       'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы',
       'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы',
       'Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара',
       'Трафарет_Рекламные показы на карточке това

In [47]:
df_result['ТипАктивности'].unique()

array(['Органика', 'Оплата за клик', 'Оплата за заказ', nan], dtype=object)

In [48]:
len(df_funnel_reference['Артикул'].unique())

65415

In [49]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-10-28"]['Артикул'].count()

np.int64(49833)

In [50]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float64(17899191.29)

In [51]:
(df_result[df_result['Дата'] == "2025-10-28"]['Вывод в топ_Расход, ₽'].sum() + df_result[df_result['Дата'] == "2025-10-28"]['Трафарет_Расход, ₽'].sum() + df_result[df_result['Дата'] == "2025-10-28"]['Оплата за заказ_Расход, ₽'].sum())

np.float32(17899192.0)

In [52]:
# df_result['Показы, всего'].sum()
df_result[df_result['Дата'] == "2025-10-28"]['Показы, всего'].sum()

np.float64(133986955.0)

In [53]:
len(df_result['Артикул'].unique())

63829

In [54]:
# df_funnel_reference_copy = df_funnel_reference.copy()
# df_funnel_reference = df_result

In [55]:
df_result.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Цена', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'Вывод в топ', 'Трафарет',
       'Оплата за заказ', 'Органика', 'Вывод в топ_Расход, ₽',
       'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽',
       'Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Ре

In [56]:
# Трафарет
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') &
    (df_result['Место размещения'] == 'Поиск и рекомендации'))
)
df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Трафарет
mask = (
    (df_result['Инструмент'] == 'Оплата за заказ')
)
df_result.loc[mask, 'ТипАктивности'] = 'Оплата за заказ'

# Вывод в топ
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') & 
     (df_result['Место размещения'] == 'Поиск'))
)
df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# Органика
mask = (
    ((df_result['Расход, ₽'] == 0) | 
     (df_result['Расход, ₽'] == 0.0))
)
df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [57]:
df_funnel_reference

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-09-01,023063J0,Органика,1,0,0,NaN,0,0,0,...,Обувь,low (L),Коновалова А.,"2019SS,2020SS",023063J0,1044.1009,0.926942,20.0,Гусева Дарья,Обувь
1,2025-09-01,02306880,Органика,1,0,0,NaN,0,0,0,...,Обувь,flat (AM),Коновалова А.,2019SS,02306880,476.5430,0.926942,20.0,Прусс Константин,Обувь
2,2025-09-01,02306990,Органика,1,0,0,NaN,0,0,0,...,Обувь,middle (AM),Коновалова А.,2019SS,02306990,545.8912,0.926942,20.0,Прусс Константин,Обувь
3,2025-09-01,023063G0,Органика,53,3,33,51.85,0,0,0,...,Обувь,wedge (L),Коновалова А.,"2019SS,2020SS",023063G0,1024.6240,1.000000,20.0,Гусева Дарья,Обувь
4,2025-09-01,b3106090,Органика,1,0,0,NaN,0,0,0,...,Обувь,flat (L),Коновалова А.,2019SS,b3106090,1461.5763,0.941667,20.0,Селютина Арина,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3801172,2025-10-31,c0101130,Органика,1,0,0,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c0101130,NaN,0.943313,20.0,NaN,NaN
3801173,2025-10-31,c1101020,Органика,3,1,1,NaN,0,0,0,...,Аксессуары,NaN,Коновалова А.,2021AW,c1101020,NaN,0.943313,20.0,Никонорова Алина,NaN
3801174,2025-10-31,u4906230,Органика,1,0,0,NaN,0,0,0,...,Игрушки,NaN,Евтикова Н.А.,"2024AW,2024SS",u4906230,303.7363,0.943561,10.0,Валиков Никита,Игрушки
3801175,2025-10-31,,Органика,1,0,0,NaN,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
df_funnel_reference = df_result

In [59]:
# funnel_columns = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
#        'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
#        'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
#        'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
#        'В корзину из карточки товара', 'Выкупили ШТ', 
#        'Расход, ₽', 'Рекламные заказано на сумму',
#        'Рекламные заказано товаров', 'Рекламные показы',
#        'Рекламные показы на карточке товара', 'Цена'
#        ]

In [60]:
# import pandas as pd
# import numpy as np

# # --- 0) подготовка
# df = df_funnel_reference.copy()

# # переименовать тип активности "ТОП" -> "Вывод в топ"
# df.loc[df['ТипАктивности'].eq('ТОП'), 'ТипАктивности'] = 'Вывод в топ'

# # полный список типов (жёстко фиксируем порядок и наличие)
# ALL_TYPES = ['Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Спецразмещение', 'Органика']

# # ключи и метрики
# key_cols = ['Дата', 'Артикул']
# # всё, что стоит в таблице ПОСЛЕ "ТипАктивности", считаем метриками
# cols = funnel_columns
# metric_cols = cols[cols.index('ТипАктивности') + 1 :]

# print(metric_cols)

# # привести метрики к числам (на случай строк/пробелов)
# for c in metric_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# # --- 1) агрегация по (Дата, Артикул, ТипАктивности)
# g = (df
#      .groupby(key_cols + ['ТипАктивности'], as_index=False)[metric_cols]
#      .sum(min_count=1)
# )

# # --- 2) «широкая» таблица метрик с префиксами <Тип>_<Метрика>
# wide_metrics = g.pivot_table(
#     index=key_cols,
#     columns='ТипАктивности',
#     values=metric_cols,
#     aggfunc='sum',
#     fill_value=0
# )

# # гарантируем наличие ВСЕХ типов и ВСЕХ метрик (даже если их не было в данных)
# full_cols = pd.MultiIndex.from_product([metric_cols, ALL_TYPES])
# wide_metrics = wide_metrics.reindex(columns=full_cols, fill_value=0)

# # имена колонок: "Тип_Метрика"
# wide_metrics.columns = [f'{act}_{met}' for met, act in wide_metrics.columns.to_flat_index()]
# wide_metrics = wide_metrics.reset_index()

# # --- 3) бинарные признаки наличия типа активности (1/0) по каждой паре (Дата, Артикул)
# presence = (
#     df.groupby(key_cols + ['ТипАктивности']).size()
#       .reset_index(name='n')
#       .pivot(index=key_cols, columns='ТипАктивности', values='n')
#       .reindex(columns=ALL_TYPES, fill_value=0)
#       .gt(0).astype(int)  # 1 если был хотя бы один ряд данного типа
#       .reset_index()
# )

# # --- 4) объединяем метрики и бинарные признаки
# out = (wide_metrics
#        .merge(presence, on=key_cols, how='left')
#        .fillna(0)
# )

# # --- 5) итоговые столбцы БЕЗ префиксов = сумма по всем типам
# for met in metric_cols:
#     to_sum = [f'{t}_{met}' for t in ALL_TYPES if f'{t}_{met}' in out.columns]
#     if to_sum:
#         out[met] = out[to_sum].sum(axis=1)

# # --- 6) порядок колонок: ключи → бинарные типы → для каждой метрики столбцы по типам → итог по метрике
# ordered = key_cols + ALL_TYPES[:]  # бинарные столбцы имеют те же имена, что и типы
# for met in metric_cols:
#     ordered += [f'{t}_{met}' for t in ALL_TYPES]
#     ordered += [met]
# # оставим только реально существующие (вдруг каких-то метрик не было)
# ordered = [c for c in ordered if c in out.columns]

# out = out[ordered]

# # результат в переменной `out`
# # одна строка на (Дата, Артикул), колоноки вида:
# # Дата | Артикул | Вывод в топ | Трафарет | ... | Вывод в топ_Показы, всего | Трафарет_Показы, всего | ... | Показы, всего | ...


In [61]:
df_funnel_reference

,Дата,Артикул,ТипАктивности,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,...,Вывод в топ_Рекламные показы,Трафарет_Рекламные показы,Оплата за заказ_Рекламные показы,Органика_Рекламные показы,Рекламные показы,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара
0,2025-09-01,e0304000,Органика,1204.0,49.0,689.0,93.10,7.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-09-01,e0304010,Органика,1065.0,60.0,439.0,112.04,8.0,4.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-09-01,c0104270,Органика,1.0,1.0,0.0,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-09-01,W2185571,Органика,1145.0,226.0,43.0,134.50,7.0,3.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-09-01,W5255315,Оплата за клик,6684.0,290.0,1190.0,103.93,15.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169860,2025-11-16,89004120,Органика,1.0,0.0,0.0,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6169861,2025-11-16,93804210,Органика,2.0,0.0,0.0,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6169862,2025-11-16,a5704000,Органика,1.0,0.0,0.0,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6169863,2025-11-16,42404090,Органика,2.0,0.0,0.0,NaN,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
df_funnel_reference[df_funnel_reference['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float32(17899192.0)

In [63]:
# import pandas as pd
# import numpy as np

# def build_funnel_wide(
#     df_raw: pd.DataFrame,
#     funnel_columns: list,
#     all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
#     infer_organic_by_zero_spend=False,
#     spend_col='Расход, ₽'
# ):
#     """
#     Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
#     добавляет итоги и ПРОНОСИТ все прочие (не суммируемые) колонки из df_raw.
#     """

#     # 0) берём только нужные колонки для расчёта метрик (экономим память/время)
#     cols_present = [c for c in funnel_columns if c in df_raw.columns]
#     df = df_raw.loc[:, cols_present].copy()

#     # 1) нормализуем типы активности
#     type_col = 'ТипАктивности'
#     df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})

#     if infer_organic_by_zero_spend and spend_col in df.columns:
#         m = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
#         df.loc[m, type_col] = 'Органика'

#     # ключи и метрики
#     key_cols = ['Дата', 'Артикул']
#     met_start = funnel_columns.index(type_col) + 1
#     metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

#     # привести метрики к числам (экономный тип)
#     for c in metric_cols:
#         df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

#     # 2) суммируем по (Дата, Артикул, ТипАктивности)
#     g = df.groupby(key_cols + [type_col], as_index=False)[metric_cols].sum(min_count=1)

#     # база ключей (одна строка на Дата+Артикул)
#     base = g[key_cols].drop_duplicates().set_index(key_cols).sort_index()

#     # --- блоки метрик по каждому типу + базовые бинарные флаги ---
#     metric_blocks, flag_blocks = [], []

#     for t in all_types:
#         sub = g[g[type_col] == t].set_index(key_cols)

#         # метрики с префиксом
#         if sub.empty:
#             sub_metrics = pd.DataFrame(
#                 0.0, index=base.index,
#                 columns=[f'{t}_{m}' for m in metric_cols],
#                 dtype='float32'
#             )
#         else:
#             sub_metrics = (sub[metric_cols]
#                            .rename(columns={m: f'{t}_{m}' for m in metric_cols})
#                            .reindex(base.index, fill_value=0.0)
#                            .astype('float32'))
#         metric_blocks.append(sub_metrics)

#         # бинарный флаг наличия типа
#         if sub.empty:
#             flag = pd.Series(0, index=base.index, name=t, dtype='int8')
#         else:
#             flag = (pd.Series(1, index=sub.index, name=t)
#                       .reindex(base.index, fill_value=0)
#                       .astype('int8'))
#         flag_blocks.append(flag)

#     metrics_block = pd.concat(metric_blocks, axis=1)
#     flags_block   = pd.concat(flag_blocks, axis=1)

#     # --- 3) ДОП. КОЛОНКИ (все из df_raw, которых нет в funnel_columns) ---
#     #    агрегируем по (Дата, Артикул) → first (первое ненулевое значение)
#     extra_cols = [c for c in df_raw.columns
#                   if c not in set(key_cols + [type_col] + metric_cols)]
#     if extra_cols:
#         # оставим только реально существующие
#         extra_cols = [c for c in extra_cols if c in df_raw.columns]
#         dims_block = (df_raw[key_cols + extra_cols]
#                         .sort_values(key_cols)
#                         .groupby(key_cols, as_index=False)
#                         .first())  # берёт первое НЕ NaN
#         # переиндексируем к базе
#         dims_block = (dims_block.set_index(key_cols)
#                                    .reindex(base.index)
#                                    .reset_index())
#     else:
#         dims_block = base.reset_index()

#     # --- 4) итоги без префиксов (сумма по всем типам) ---
#     totals = pd.DataFrame(index=base.index)
#     for m in metric_cols:
#         totals[m] = metrics_block[[f'{t}_{m}' for t in all_types]].sum(axis=1).astype('float32')

#     # --- 5) финальная сборка ---
#     out = pd.concat(
#         [
#             dims_block,  # ключи + все доп. колонки
#             flags_block.reset_index(drop=True),
#             metrics_block.reset_index(drop=True),
#             totals.reset_index(drop=True)
#         ],
#         axis=1
#     ).copy()

#     # --- 6) порядок колонок: ключи → доп.колонки → флаги → метрики по типам → итоги ---
#     ordered = []
#     # ключи
#     ordered += key_cols
#     # доп. колонки в исходном порядке df_raw (после ключей)
#     ordered += [c for c in df_raw.columns
#                 if c in out.columns and c not in key_cols + [type_col] + metric_cols]
#     # флаги типов
#     ordered += [t for t in all_types if t in out.columns]
#     # метрики по типам + итог без префикса
#     for m in metric_cols:
#         ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
#         if m in out.columns:
#             ordered += [m]

#     out = out[[c for c in ordered if c in out.columns]]

#     return out


In [64]:
# out = build_funnel_wide(df_raw=df_funnel_reference, funnel_columns=funnel_columns)
# out

In [65]:
out[out['Дата'] == "2025-10-20"]['Расход, ₽'].sum()

np.float32(10009251.0)

In [66]:
out.columns

Index(['Дата', 'Артикул OZ', 'ID кампании', 'Инструмент', 'Место размещения',
       'Артикул', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика',
       'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽',
       'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽',
       'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Рекламные заказано на сумму',
       'Оплата за заказ_Рекламные заказано на сумму',
       'Органика_Рекламные заказано на сумму', 'Рекламные заказано на сумму',
       'Вывод в топ_Рекламные заказано товаров',
       'Трафарет_Рекламные заказано товаров',
       'Оплата за заказ_Рекламные заказано товаров',
       'Органика_Рекламные заказано товаров', 'Рекламные заказано товаров',
       'Вывод в топ_Рекламные показы', 'Трафарет_Рекламные показы',
       'Оплата за заказ_Рекламные показы', 'Органика_Рекламные показы',
       'Рекламные показы', 'Вывод в топ_Рекламные показы на карточке товара',
       'Трафарет_Рекламные показы на карточке това

In [67]:
# df_funnel_reference = out

In [68]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Цена', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'Вывод в топ', 'Трафарет',
       'Оплата за заказ', 'Органика', 'Вывод в топ_Расход, ₽',
       'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽',
       'Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Ре

In [69]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата   Артикул   ТипАктивности  Показы, всего  \
0  2025-09-01  e0304000        Органика         1204.0   
1  2025-09-01  e0304010        Органика         1065.0   
2  2025-09-01  c0104270        Органика            1.0   
3  2025-09-01  W2185571        Органика         1145.0   
4  2025-09-01  W5255315  Оплата за клик         6684.0   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                       49.0                       689.0   
1                       60.0                       439.0   
2                        1.0                         0.0   
3                      226.0                        43.0   
4                      290.0                      1190.0   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                        93.10               7.0               0.0   
1                       112.04               8.0               4.0

In [70]:
df_final_db.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Цена', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'Вывод в топ', 'Трафарет',
       'Оплата за заказ', 'Органика', 'Вывод в топ_Расход, ₽',
       'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽',
       'Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Ре

In [71]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="openpyxl")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="openpyxl")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [72]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата   Артикул   ТипАктивности  Показы, всего  \
0  2025-09-01  e0304000        Органика         1204.0   
1  2025-09-01  e0304010        Органика         1065.0   
2  2025-09-01  c0104270        Органика            1.0   
3  2025-09-01  W2185571        Органика         1145.0   
4  2025-09-01  W5255315  Оплата за клик         6684.0   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                       49.0                       689.0   
1                       60.0                       439.0   
2                        1.0                         0.0   
3                      226.0                        43.0   
4                      290.0                      1190.0   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                        93.10               7.0               0.0   
1                       112.04               8.0    

In [73]:
del df_item_features

In [74]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    # Сохранение финальной таблицы
    # df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата   Артикул   ТипАктивности  Показы, всего  \
0  2025-09-01  e0304000        Органика         1204.0   
1  2025-09-01  e0304010        Органика         1065.0   
2  2025-09-01  c0104270        Органика            1.0   
3  2025-09-01  W2185571        Органика         1145.0   
4  2025-09-01  W5255315  Оплата за клик         6684.0   

   Показы на карточке товара  Показы в поиске и каталоге  \
0                       49.0                       689.0   
1                       60.0                       439.0   
2                        1.0                         0.0   
3                      226.0                        43.0   
4                      290.0                      1190.0   

   Позиция в поиске и каталоге  В корзину, всего  Заказано товаров  \
0                        93.10               7.0               0.0   
1                       112.04               8.0               4.0  

In [75]:
import numpy as np
type_order_paid = ['Вывод в топ', 'Трафарет', 'Оплата за заказ']
paid_cols = [c for c in type_order_paid if c in df_final_db_all_features.columns]  # на случай отсутствующих

flags = df_final_db_all_features[paid_cols].fillna(0).astype('uint8').to_numpy()
labels = np.array(paid_cols, dtype=object)

combo = ['/'.join(labels[row.astype(bool)]) if row.any() else '' for row in flags]
df_final_db_all_features['ТипАктивности'] = combo

# Если есть только органика — подставим "Органика"
if 'Органика' in df_final_db_all_features.columns:
    only_org = df_final_db_all_features['Органика'].fillna(0).astype('uint8').eq(1) & (flags.sum(axis=1) == 0)
    df_final_db_all_features.loc[only_org, 'ТипАктивности'] = 'Органика'

# Пустые — на "—"
df_final_db_all_features['ТипАктивности'] = df_final_db_all_features['ТипАктивности'].replace('', '—')

In [76]:
# Органика
mask = (
    ((df_final_db_all_features['Расход, ₽'] == 0) | 
     (df_final_db_all_features['Расход, ₽'] == 0.0))
)
df_final_db_all_features.loc[mask, 'ТипАктивности'] = 'Органика'

In [77]:
df_final_db_all_features[df_final_db_all_features['Дата'] == "2025-10-28"]['Расход, ₽'].sum()

np.float32(17899192.0)

In [78]:
# === SQL СЦЕПКИ ОЗОН ===
sql = """
SELECT scepka.[id]
      ,scepka.[offer_id]
      ,scepka.[product_id]
      ,sku.fbo_sku as [Артикул OZ]
      ,scepka.[group_value] as [Текущая склейка]
      ,sku.[article]
      ,scepka.[updated_at] as [Дата Обновления]
  FROM [DBReport].[mp].[ozon_scepka] scepka
  JOIN [DBReport].[mp].[ozon_sku] sku 
  ON  scepka.[product_id] = sku.[product_id] 
  and sku.actual = 1
"""
df_links = pd.read_sql(sql, engine)
df_links['Артикул OZ'] = df_links['Артикул OZ'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\OZ\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_OZ.xlsx"))

In [79]:
df_links['Текущая склейка'].unique()

array(['W8429001', '1716', '862', ..., '2623_M7455670-45',
       '2623_M7459003-44', None], shape=(30388,), dtype=object)

In [80]:
df_final_db_all_features = pd.merge(df_final_db_all_features, df_links[["Артикул OZ", "Текущая склейка"]], how='left', on='Артикул OZ')

In [81]:
# df_final_db_all_features_new['Текущая склейка'].unique()

In [82]:
df_final_db_all_features.columns

Index(['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
       'Показы на карточке товара', 'Показы в поиске и каталоге',
       'Позиция в поиске и каталоге', 'В корзину, всего', 'Заказано товаров',
       'Отменено товаров', 'Доставлено товаров', 'Возвращено товаров',
       'Заказано на сумму', 'В корзину из карточки товара', 'Выкупили ШТ',
       'Цена', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
       'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа',
       'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения', 'Вывод в топ', 'Трафарет',
       'Оплата за заказ', 'Органика', 'Вывод в топ_Расход, ₽',
       'Трафарет_Расход, ₽', 'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽',
       'Расход, ₽', 'Вывод в топ_Рекламные заказано на сумму',
       'Трафарет_Ре

In [83]:
count = 0
for item in list(df_final_db_all_features.columns):
    print(f'{{"{item}", type {str(type(df_final_db_all_features[item].unique()[0]))}}}, ', end="")
    count +=1
    if count == 5:
        print("\n", end="")
        count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"ТипАктивности", type <class 'str'>}, {"Показы, всего", type <class 'numpy.float64'>}, {"Показы на карточке товара", type <class 'numpy.float64'>}, 
{"Показы в поиске и каталоге", type <class 'numpy.float64'>}, {"Позиция в поиске и каталоге", type <class 'numpy.float64'>}, {"В корзину, всего", type <class 'numpy.float64'>}, {"Заказано товаров", type <class 'numpy.float64'>}, {"Отменено товаров", type <class 'numpy.float64'>}, 
{"Доставлено товаров", type <class 'numpy.float64'>}, {"Возвращено товаров", type <class 'numpy.float64'>}, {"Заказано на сумму", type <class 'numpy.float64'>}, {"В корзину из карточки товара", type <class 'numpy.float64'>}, {"Выкупили ШТ", type <class 'numpy.float64'>}, 
{"Цена", type <class 'numpy.float64'>}, {"Артикул OZ", type <class 'str'>}, {"Наименование", type <class 'str'>}, {"Коллекция", type <class 'str'>}, {"Бренд", type <class 'str'>}, 
{"Сезон", type <class 'str'>}, {"Направление", type

In [84]:
print(len(list(df_final_db_all_features.columns)))

80


In [85]:
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)

In [86]:
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"))


In [ ]:
telegram_sendMessage(chat_id=421762273, text="ОЗОН CSV тест. готов. {}")

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
# Сохранение финальной таблицы
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon.csv"), index=False, chunksize=500_000, engine='pyarrow')

In [ ]:
# df_final_db_all_features[((df_final_db_all_features['Расход, ₽'] == 0) & (df_final_db_all_features['ТипАктивности'] == "Трафарет"))][['Артикул', "Дата", "Артикул OZ", "ТипАктивности", "Инструмент", "Расход, ₽"]]

In [ ]:
# del df_date_features
# gc.collect()

In [87]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [88]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)
        # success = True

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку ' {FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...
Файл 'Показы и затраты ОЗ_2.0 14.11.xlsx' удален из папки '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл 'Показы и затраты ОЗ_2.0 14.11.xlsx' удален из папки '\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям'.
Попытка 1 обновить файл 'Показы и затраты ОЗ_2.0.xlsx'...
Выполняем обновление данных...
Файл успешно сохранен с оригинальным именем в '\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям'.
Файл успешно сохранен как 'Показы и затраты ОЗ_2.0 17.11.xlsx'.
Файл успешно скопирован в папку '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл успешно обновлен и сохранен. Время выполнения: 0 часа(ов) 0 минут(ы) 26.49 секунд


In [98]:
telegram_sendMessage(chat_id=421762273, text="ОЗОН Дашборд тест. готов.")

RuntimeError: asyncio.run() cannot be called from a running event loop